# Joga candidate boundary pipeline check
Private Kaggle CPU. Internet On. No model training, labels or study-area approval.


In [ ]:
BOUNDARY = None


In [ ]:
"""Fresh Phase 0 experiment: bounded real-data inspection in a hosted runtime."""
import hashlib
import importlib.metadata
import json
import math
import platform
import shutil
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

COLLECTION = 'sentinel-2-c1-l2a'
SCENE = 'S2C_T43QFE_20250329T053314_L2A'
BBOX = [76.785, 22.413, 76.805, 22.433]
BANDS = [('blue', 'B02'), ('green', 'B03'), ('red', 'B04'), ('nir', 'B08')]


def run(bbox=None, area_label='provisional research crop; not the Joga beat boundary', scene_id=SCENE):
    if platform.system() == 'Windows':
        raise RuntimeError('Use a hosted Kaggle or Colab CPU runtime; local processing is disabled.')
    if Path('/kaggle/working').is_dir():
        base, provider = Path('/kaggle/working'), 'Kaggle'
    elif Path('/content').is_dir():
        try:
            import google.colab
        except ImportError as error:
            raise RuntimeError('Use a hosted Colab runtime.') from error
        base, provider = Path('/content'), 'Colab'
    else:
        raise RuntimeError('Use a hosted Kaggle or Colab CPU runtime; local processing is disabled.')

    bbox = BBOX if bbox is None else list(bbox)
    if (len(bbox) != 4 or not all(math.isfinite(v) for v in bbox)
            or not -180 <= bbox[0] < bbox[2] <= 180
            or not -90 <= bbox[1] < bbox[3] <= 90):
        raise ValueError('Provide west/south/east/north bounds in longitude/latitude degrees.')

    packages = {}
    for name in ['numpy', 'rasterio', 'Pillow']:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError as error:
            raise RuntimeError(f'Cloud package missing: {name}. No automatic installation performed.') from error
    import numpy as np
    import rasterio
    from PIL import Image
    from rasterio.vrt import WarpedVRT
    from rasterio.warp import Resampling, transform as project_points, transform_bounds
    from rasterio.windows import Window, from_bounds

    if shutil.disk_usage(base).free < 100 * 1024**2:
        raise RuntimeError('Cloud runtime needs at least 100 MiB free for this bounded experiment.')
    output = base / 'forestguard_phase0' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output.mkdir(parents=True)
    started = time.monotonic()
    if not isinstance(scene_id, str) or not scene_id.startswith('S2') or not all(c.isalnum() or c == '_' for c in scene_id):
        raise ValueError('Provide a Sentinel-2 scene identifier.')
    url = f'https://earth-search.aws.element84.com/v1/collections/{COLLECTION}/items/{scene_id}'
    try:
        with urlopen(url, timeout=30) as response:
            content = response.read(512001)
    except URLError as error:
        raise RuntimeError('Metadata request failed. Check cloud Internet and any account verification requirement.') from error
    if len(content) > 512000:
        raise ValueError('Metadata exceeds the 500 KiB safety limit.')
    item = json.loads(content)
    assert item['id'] == scene_id
    (output / 'source.json').write_bytes(content)
    assets = item['assets']
    calibration, measurements, masks = {}, [], []
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', GDAL_HTTP_TIMEOUT='60',
                      CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif', GDAL_CACHEMAX=64*1024**2):
        with rasterio.open(assets['red']['href']) as reference:
            bounds = transform_bounds('EPSG:4326', reference.crs, *bbox, densify_pts=21)
            crop = from_bounds(*bounds, transform=reference.transform)
            left, top = math.floor(crop.col_off), math.floor(crop.row_off)
            right, bottom = math.ceil(crop.col_off+crop.width), math.ceil(crop.row_off+crop.height)
            assert 0 <= left < right <= reference.width and 0 <= top < bottom <= reference.height
            assert max(right-left, bottom-top) <= 512, 'Crop exceeds the research size limit'
            window = Window(left, top, right-left, bottom-top)
            crs, grid = reference.crs, reference.window_transform(window)
            full_grid = reference.transform
            shape = (bottom-top, right-left)
        for key, name in BANDS:
            asset = assets[key]
            assert asset['href'].startswith('https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/')
            scale = asset['raster:bands'][0]['scale']
            offset = asset['raster:bands'][0]['offset']
            assert math.isfinite(scale) and scale > 0 and math.isfinite(offset)
            with rasterio.open(asset['href']) as band:
                assert band.crs == crs and band.transform == full_grid and band.res == (10., 10.)
                assert math.isclose(band.scales[0],scale) and math.isclose(band.offsets[0],offset), 'Calibration metadata/header mismatch'
                raw = band.read(1, window=window, masked=True)
            calibration[name] = {'scale': scale, 'offset': offset, 'href': asset['href']}
            measurements.append(raw.filled(0).astype('float32') * scale + offset)
            masks.append(~np.ma.getmaskarray(raw))
        with rasterio.open(assets['scl']['href']) as quality:
            with WarpedVRT(quality, crs=crs, transform=grid, height=shape[0], width=shape[1],
                           resampling=Resampling.nearest) as aligned:
                scl = aligned.read(1)

    values = np.stack(measurements)
    row, col = np.indices(shape)
    x, y = grid.c+(col+.5)*grid.a, grid.f+(row+.5)*grid.e
    lon, lat = project_points(crs, 'EPSG:4326', x.ravel(), y.ravel())
    lon, lat = np.asarray(lon).reshape(shape), np.asarray(lat).reshape(shape)
    study = (lon >= bbox[0]) & (lon <= bbox[2]) & (lat >= bbox[1]) & (lat <= bbox[3])
    valid = study & np.logical_and.reduce(masks) & np.isfinite(values).all(axis=0) & np.isin(scl, [4, 5, 6])
    assert valid.any(), 'No usable pixels: inspect another observation or research crop'
    values[:, ~valid] = -9999
    profile = dict(driver='GTiff', crs=crs, transform=grid, height=shape[0], width=shape[1], compress='deflate')
    with rasterio.open(output/'reflectance.tif', 'w', **profile, count=4, dtype='float32', nodata=-9999) as dst:
        dst.write(values)
        dst.descriptions = tuple(name for _, name in BANDS)
    for name, array in [('scl', scl), ('study', study.astype('uint8')), ('usable', valid.astype('uint8'))]:
        with rasterio.open(output/f'{name}.tif', 'w', **profile, count=1, dtype='uint8') as dst:
            dst.write(array, 1)
    rgb = (np.clip(values[[2, 1, 0]] / .3, 0, 1)*255).astype('uint8').transpose(1, 2, 0)
    Image.fromarray(np.dstack([rgb, valid.astype('uint8')*255])).save(output/'preview.png')

    # Reopen outputs: successful writes alone are insufficient evidence.
    for name, expected in [('reflectance', values), ('scl', scl), ('study', study), ('usable', valid)]:
        with rasterio.open(output/f'{name}.tif') as saved:
            assert saved.crs == crs and saved.transform == grid and saved.shape == shape
            assert saved.res == (10., 10.)
            actual = saved.read() if expected.ndim == 3 else saved.read(1)
            assert np.array_equal(actual, expected), f'Saved pixels differ: {name}'
    classes, counts = np.unique(scl[study], return_counts=True)
    report = {
        'status': 'real research sample; not forest labels or beat reporting',
        'scene_id': scene_id, 'collection':COLLECTION, 'acquisition': item['properties']['datetime'], 'metadata_url': url,
        'bbox_lon_lat': bbox, 'area_label':area_label, 'official_boundary_verified': False, 'shape': list(shape),
        'crs': str(crs), 'transform': list(grid)[:6], 'resolution_m': 10,
        'band_order': [name for _, name in BANDS], 'calibration': calibration,
        'reflectance_formula': 'raw * asset scale + asset offset, applied once',
        'calibration_limit': 'Collection 1 asset metadata matches all four raster headers; independent absolute radiometric validation not performed.',
        'quality_rule': 'all bands valid and finite; SCL 4/5/6; inside research-box pixel centers',
        'scl_resampling': 'nearest neighbor; native 20 m quality remains 20 m information',
        'study_pixels': int(study.sum()), 'usable_pixels': int(valid.sum()),
        'usable_fraction': float(valid.sum()/study.sum()),
        'scl_counts_inside_study': {str(int(k)): int(v) for k, v in zip(classes, counts)},
        'reflectance_quantiles': {name: np.quantile(values[i, valid], [0, .01, .5, .99, 1]).tolist()
                                 for i, (_, name) in enumerate(BANDS)},
        'runtime': {'provider': provider, 'python': platform.python_version(), 'packages': packages,
                    'gdal': rasterio.__gdal_version__},
        'processing_seconds': round(time.monotonic()-started, 3),
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'attribution': f'Contains modified Copernicus Sentinel data {item["properties"]["datetime"][:4]}',
        'license_url': 'https://cds.climate.copernicus.eu/licences/ec-sentinel',
        'model_trained': False, 'reviewed_labels_exist': False,
    }
    (output/'report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    manifest = {}
    for file in sorted(output.iterdir()):
        with file.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        manifest[file.name] = {'sha256': digest, 'bytes': file.stat().st_size}
    (output/'checksums.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    bundle = output/'forestguard_phase0.zip'
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name in [*manifest, 'checksums.json']:
            archive.write(output/name, arcname=name)
    with zipfile.ZipFile(bundle) as archive:
        assert archive.testzip() is None
        for name, record in manifest.items():
            assert hashlib.sha256(archive.read(name)).hexdigest() == record['sha256']
    print(json.dumps(report, indent=2))
    print('Verified sample bundle:', bundle, '\nBundle bytes:', bundle.stat().st_size)
    return output




In [ ]:

if BOUNDARY is None:
    raise RuntimeError('Supply the authorized pipeline-check candidate in a private notebook.')
feature = BOUNDARY['features'][0]
assert feature['properties']['pilot_approved'] is False
output = run(bbox=BOUNDARY['bbox'], area_label='Joga 278 candidate: pipeline checks only; not approved study area',
             scene_id='S2A_T43QFE_20250328T052953_L2A')
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.features import geometry_mask
from rasterio.warp import transform_geom
from PIL import Image
report = json.loads((output/'report.json').read_text())
with rasterio.open(output/'reflectance.tif') as raster:
    geometry = transform_geom('EPSG:4326', raster.crs, feature['geometry'])
    inside = geometry_mask([geometry], out_shape=raster.shape, transform=raster.transform,
                           invert=True, all_touched=False)
    assert inside.any()
    valid = raster.read(4) != raster.nodata
    usable = inside & valid
    profile = raster.profile.copy()
    profile.update(count=1, dtype='uint8', nodata=None)
    with rasterio.open(output/'boundary_mask.tif', 'w', **profile) as dst:
        dst.write(inside.astype('uint8'),1)
    with rasterio.open(output/'boundary_mask.tif') as saved:
        assert saved.transform == raster.transform and saved.crs == raster.crs
        assert np.array_equal(saved.read(1),inside)
    fig,ax = plt.subplots(figsize=(7,7))
    ax.imshow(Image.open(output/'preview.png'))
    inverse = ~raster.transform
    area_m2, perimeter_m = 0., 0.
    for polygon in geometry['coordinates']:
        for index,ring in enumerate(polygon):
            points = np.asarray(ring)[:,:2]
            shifted = points-points[0]
            signed = .5*np.sum(shifted[:-1,0]*shifted[1:,1]-shifted[1:,0]*shifted[:-1,1])
            area_m2 += abs(signed)*(1 if index == 0 else -1)
            perimeter_m += np.linalg.norm(np.diff(points,axis=0),axis=1).sum()
            pixels = np.asarray([inverse*(x,y) for x,y in points])
            ax.plot(pixels[:,0]-.5,pixels[:,1]-.5,color='yellow',linewidth=1.2)
    assert area_m2 > 0
    pixel_area_m2 = abs(raster.transform.a*raster.transform.e)
    raster_area_m2 = int(inside.sum())*pixel_area_m2
    assert abs(raster_area_m2-area_m2) <= perimeter_m*10, 'Rasterized area exceeds boundary-pixel tolerance'
    ax.set_title('Official-source Joga 278 candidate on 2025-03-28 imagery\nPipeline check only; no forest labels or boundary approval')
    ax.set_xlabel('10 m raster column'); ax.set_ylabel('10 m raster row')
    fig.tight_layout(); fig.savefig(output/'boundary_overlay.png',dpi=150)
    plt.show(); plt.close(fig)
    report.update(candidate_pixels=int(inside.sum()),candidate_usable_pixels=int(usable.sum()),
        candidate_usable_fraction=float(usable.sum()/inside.sum()),
        candidate_geometric_area_ha=float(area_m2/10000),candidate_pixel_area_ha=float(raster_area_m2/10000),
        candidate_perimeter_m=float(perimeter_m),source_area_attribute_ha=feature['properties']['AREA_HA'],
        historical_area_attribute_ha=580.770,area_kind='candidate geometry/coverage; not forest area or beat totals',
        polygon_mask_rule='pixel centers; all_touched=False',pilot_approved=False,
        current_beat_boundary_verified=False,positional_accuracy_measured=False,
        label_feasibility='Requires independent dated review; no forest classes assigned',
        boundary_source_sha256=feature['properties']['source_sha256'],
        matplotlib_version=importlib.metadata.version('matplotlib'))
(output/'boundary_input.geojson').write_text(json.dumps(BOUNDARY,indent=2))
(output/'boundary_review.json').write_text(json.dumps(report,indent=2))
names = ['boundary_input.geojson','boundary_mask.tif','boundary_overlay.png','boundary_review.json']
manifest = {name:{'bytes':(output/name).stat().st_size,'sha256':hashlib.sha256((output/name).read_bytes()).hexdigest()} for name in names}
(output/'boundary_checksums.json').write_text(json.dumps(manifest,indent=2))
with zipfile.ZipFile(output/'boundary_review.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for name in names+['boundary_checksums.json']:
        archive.write(output/name,arcname=name)
with zipfile.ZipFile(output/'boundary_review.zip') as archive:
    assert archive.testzip() is None
    for name,record in manifest.items():
        assert hashlib.sha256(archive.read(name)).hexdigest()==record['sha256']
print(json.dumps(report,indent=2))
print('Verified boundary-check export:',output/'boundary_review.zip')
